# LLM Model Merge & GGUF Conversion

이 노트북은 **4개의 모델**을 GGUF 포맷으로 변환하기 위해 작성되었습니다.

**대상 모델 리스트:**
1. **Base Model**: Meta-Llama-3-8B (변환만 진행)
2. **Adapter 1**: Constant Schedule (Step 100)
3. **Adapter 2**: Cosine Schedule (Step 100)
4. **Adapter 3**: Cosine Schedule (Step 1000)

---

## 1. 환경 설정 및 라이브러리 설치
필요한 라이브러리를 임포트하고 `llama.cpp`를 준비합니다.

In [ ]:
# [중요] CUDA OOM 방지: 강제로 CPU만 사용하도록 설정
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "" # GPU 비활성화

import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# llama.cpp가 없다면 클론 (GGUF 변환 도구)
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp
    
# 주의: llama.cpp의 requirements.txt를 통째로 설치하면 torch 버전이 충돌날 수 있으므로,
# 변환에 꼭 필요한 라이브러리만 안전하게 설치합니다. (이미 환경에 설치되어 있다면 생략됨)
!pip install gguf sentencepiece protobuf --no-deps

## 2. 모델 경로 설정 (Configuration)
**[중요]** 사용자 확인 완료된 경로로 설정되었습니다.

In [ ]:
# 기본 모델 ID
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B"

# 변환할 모델 리스트 설정
# type: "base" (베이스 모델 단독) 또는 "adapter" (병합 필요)
# path: 어댑터 경로 (base인 경우 저장될 폴더명)
# name: 최종 저장될 GGUF 파일명 접두사

TARGET_MODELS = [
    # 1. Base Model (병합 없음)
    {
        "type": "base",
        "name": "llama3-8b-base",
        "path": "./merged_models/base_model_fp16"
    },
    # 2. Adapter 1: Constant Step 100
    {
        "type": "adapter",
        "name": "llama3-8b-constant-100",
        "path": "./output" # 사용자 확인: output 폴더 자체에 저장됨
    },
    # 3. Adapter 2: Cosine Step 100
    {
        "type": "adapter",
        "name": "llama3-8b-cosine-100",
        "path": "./output/checkpoint-100" # 사용자 확인: checkpoint-100
    },
    # 4. Adapter 3: Cosine Step 1000
    {
        "type": "adapter",
        "name": "llama3-8b-cosine-1000",
        "path": "./output/checkpoint-1000"
    }
]

# 결과물이 저장될 폴더 생성
os.makedirs("./merged_models", exist_ok=True)
os.makedirs("./gguf_models", exist_ok=True)

## 3. 모델 병합 및 변환 함수 정의
반복적인 작업을 수행하기 위한 함수입니다.

In [ ]:
def process_model(target_info):
    model_type = target_info["type"]
    name = target_info["name"]
    path = target_info["path"]
    
    print(f"========== Processing: {name} ({model_type}) ==========")
    
    # 1. 모델 로드 및 병합 (또는 베이스 모델 저장)
    save_path = f"./merged_models/{name}"
    
    if model_type == "adapter":
        print(f"Loading Base Model: {BASE_MODEL_ID}...")
        # device_map="cpu"를 사용하여 강제로 CPU 로드
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            low_cpu_mem_usage=True,
            return_dict=True,
            torch_dtype=torch.float16,
            device_map="cpu"
        )
        
        print(f"Merging Adapter from {path}...")
        model = PeftModel.from_pretrained(base_model, path)
        model = model.merge_and_unload()
        
        print(f"Saving Merged Model to {save_path}...")
        model.save_pretrained(save_path)
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
        tokenizer.save_pretrained(save_path)
        del model, base_model 
        
    elif model_type == "base":
        print(f"Loading Base Model to save locally: {BASE_MODEL_ID}...")
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            low_cpu_mem_usage=True,
            torch_dtype=torch.float16,
            device_map="cpu"
        )
        model.save_pretrained(save_path)
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
        tokenizer.save_pretrained(save_path)
        del model

    # 메모리 정리
    gc.collect()

    # 2. GGUF 변환 (llama.cpp)
    # 최신 llama.cpp는 convert.py 대신 convert_hf_to_gguf.py를 사용합니다.
    print(f"Converting to GGUF...")
    gguf_filename = f"./gguf_models/{name}.gguf"
    
    # 변환 스크립트 실행 (최신 버전 대응)
    if os.path.exists("llama.cpp/convert_hf_to_gguf.py"):
        !python llama.cpp/convert_hf_to_gguf.py {save_path} --outfile {gguf_filename} --outtype f16
    else:
        !python llama.cpp/convert.py {save_path} --outfile {gguf_filename} --outtype f16
    
    print(f"Done! Output: {gguf_filename}\n")

## 4. 전체 실행
설정된 모든 모델에 대해 변환을 수행합니다.

In [ ]:
for target in TARGET_MODELS:
    try:
        process_model(target)
    except Exception as e:
        print(f"Error processing {target['name']}: {e}")